# Отток клиентов банка — Random Forest

[Churn Modelling (Kaggle)](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling).


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Use scikit-learn==1.6.1 and restart the kernel.")
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
DATA_URL = "https://raw.githubusercontent.com/selva86/datasets/master/Churn_Modelling.csv"
data_path = ROOT / "data" / "Churn_Modelling.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)
if not data_path.exists():
    urlretrieve(DATA_URL, data_path)
df = pd.read_csv(data_path)
print("Dataset shape:", df.shape)


Dataset shape: (10000, 14)


## Входные признаки и целевая переменная

Exited=1 означает уход клиента. Исключаем RowNumber, CustomerId, Surname и сам Exited из входа. Оставляем 10 признаков. Разделение 80/20 со стратификацией; обработка обучается только на тренировочной выборке.


In [2]:
features = ["CreditScore", "Geography", "Gender", "Age", "Tenure", "Balance",
            "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"]
X = df[features].copy()
y = df["Exited"]
assert not X.isna().any().any()
assert set(y.unique()) == {0, 1}
assert set(X["Geography"].unique()) == {"France", "Germany", "Spain"}
assert set(X["Gender"].unique()) == {"Female", "Male"}
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="error", sparse_output=False),
     ["Geography", "Gender"]),
], remainder="passthrough")
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=1
    )),
])
pipeline.fit(X_train, y_train)
assert list(pipeline.classes_) == [0, 1]
threshold = 0.5
scores = pipeline.predict_proba(X_test)[:, 1]
metrics = {
    "roc_auc": float(roc_auc_score(y_test, scores)),
    "accuracy": float(accuracy_score(y_test, scores >= threshold)),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, scores >= threshold, zero_division=0))


{
  "roc_auc": 0.8564342462647546,
  "accuracy": 0.8655
}
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.81      0.45      0.58       407

    accuracy                           0.87      2000
   macro avg       0.84      0.71      0.75      2000
weighted avg       0.86      0.87      0.85      2000



## Сохранение

Сохраняем pipeline вместе с метаданными. 


In [3]:
metadata = {
    "features": features,
    "model_version": "bank-random-forest-1.0",
    "threshold": threshold,
    "sklearn_version": sklearn.__version__,
    "metrics": metrics,
    "dataset": "Churn Modelling (bank customers)",
    "dataset_page": "https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling",
    "data_source": DATA_URL,
    "data_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
    "target": "Exited",
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "random_state": 42,
}
output_path = ROOT / "artifact" / "bank_churn.joblib"
output_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipeline, "metadata": metadata}, output_path)
output_path.with_suffix(".metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
loaded = joblib.load(output_path)
np.testing.assert_allclose(
    loaded["pipeline"].predict_proba(X_test.iloc[:5]),
    pipeline.predict_proba(X_test.iloc[:5]),
)
print("Saved and verified:", output_path)


Saved and verified: C:\Users\Yanochka\Documents\ML_PRO2026\artifact\bank_churn.joblib
